# 🚛 Detecção de Placas de Caminhão — YOLOv8 + OCR

**Fluxo:**
1. Instalar dependências
2. Baixar dataset do Roboflow
3. Treinar YOLOv8
4. Rodar detecção em vídeo com OCR + alerta sonoro

> ⚠️ Certifique-se de estar usando GPU: `Ambiente → Alterar tipo de ambiente → T4 GPU`

## 1️⃣ Instalar dependências

In [ ]:
!pip install ultralytics roboflow easyocr opencv-python-headless Pillow
import torch
print('GPU disponível:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2️⃣ Baixar dataset do Roboflow

> Cole aqui o código que o Roboflow vai te dar ao exportar no formato YOLOv8.

In [ ]:
# ⬇️ SUBSTITUA pelo código gerado no Roboflow (Export → YOLOv8 → Show download code)
from roboflow import Roboflow

rf = Roboflow(api_key="COLE_SUA_API_KEY_AQUI")
project = rf.workspace("COLE_SEU_WORKSPACE").project("COLE_SEU_PROJETO")
dataset = project.version(1).download("yolov8")

print('Dataset baixado em:', dataset.location)

## 3️⃣ Treinar o modelo YOLOv8

In [ ]:
from ultralytics import YOLO

# Carrega modelo base (yolov8n = nano, mais rápido; yolov8s = small, mais preciso)
model = YOLO('yolov8n.pt')

# Treina
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=50,         # aumente para 100 se tiver tempo
    imgsz=640,
    batch=16,
    name='placas_caminhao',
    patience=10,       # para cedo se não melhorar
    device=0           # usa GPU
)

print('Treinamento concluído!')
print('Melhor modelo salvo em: runs/detect/placas_caminhao/weights/best.pt')

## 4️⃣ Avaliar o modelo treinado

In [ ]:
# Métricas no conjunto de validação
metrics = model.val()
print(f'mAP50: {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')

# Visualizar curvas de treino
from IPython.display import Image
Image('runs/detect/placas_caminhao/results.png')

## 5️⃣ Fazer upload do vídeo (.mp4 ou .dav)

In [ ]:
from google.colab import files
import os

print('Selecione seu arquivo de vídeo (.mp4 ou .dav)...')
uploaded = files.upload()

video_path = list(uploaded.keys())[0]
print(f'Vídeo carregado: {video_path}')

# Se for .dav, converte para .mp4 primeiro
if video_path.endswith('.dav'):
    !ffmpeg -i "{video_path}" -c copy video_convertido.mp4 -y
    video_path = 'video_convertido.mp4'
    print('Convertido para MP4:', video_path)

## 6️⃣ Detecção no vídeo com OCR + Alerta Sonoro

In [ ]:
import cv2
import easyocr
import numpy as np
from ultralytics import YOLO
from IPython.display import display, HTML
from google.colab.patches import cv2_imshow
import time
import base64

# ── Carrega modelo treinado ──────────────────────────────────────────────────
MODEL_PATH = 'runs/detect/placas_caminhao/weights/best.pt'
model = YOLO(MODEL_PATH)
reader = easyocr.Reader(['pt', 'en'], gpu=torch.cuda.is_available())

# ── Função de alerta sonoro (beep via JS no Colab) ───────────────────────────
def alerta_sonoro():
    display(HTML("""
    <script>
    var ctx = new (window.AudioContext || window.webkitAudioContext)();
    var osc = ctx.createOscillator();
    var gain = ctx.createGain();
    osc.connect(gain);
    gain.connect(ctx.destination);
    osc.frequency.value = 880;
    gain.gain.setValueAtTime(0.3, ctx.currentTime);
    gain.gain.exponentialRampToValueAtTime(0.001, ctx.currentTime + 0.4);
    osc.start(ctx.currentTime);
    osc.stop(ctx.currentTime + 0.4);
    </script>
    """))

# ── Função principal de detecção ─────────────────────────────────────────────
def detectar_placas_video(video_path, confianca=0.5, mostrar_frames=True, max_frames=None):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print('❌ Erro ao abrir vídeo:', video_path)
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f'📹 Vídeo: {total_frames} frames | {fps:.1f} FPS')

    placas_detectadas = []
    ultimo_alerta = 0
    intervalo_alerta = 3  # segundos entre alertas
    frame_count = 0

    # Saída de vídeo
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter('resultado_deteccao.mp4', fourcc, fps,
                          (int(cap.get(3)), int(cap.get(4))))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if max_frames and frame_count >= max_frames:
            break

        frame_count += 1

        # Detecta a cada 3 frames (mais rápido)
        if frame_count % 3 != 0:
            out.write(frame)
            continue

        # ── YOLOv8 detecção ──
        results = model(frame, conf=confianca, verbose=False)[0]
        frame_anotado = frame.copy()

        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])

            # Recorta a região da placa
            roi = frame[y1:y2, x1:x2]
            if roi.size == 0:
                continue

            # ── OCR na placa ──
            try:
                ocr_result = reader.readtext(roi)
                texto_placa = ' '.join([r[1] for r in ocr_result if r[2] > 0.3])
                texto_placa = texto_placa.upper().strip()
            except:
                texto_placa = ''

            # ── Desenha bounding box ──
            cor = (0, 255, 0)  # verde
            cv2.rectangle(frame_anotado, (x1, y1), (x2, y2), cor, 2)

            # ── Notificação com texto OCR ──
            label = f'PLACA {conf:.0%}'
            if texto_placa:
                label += f' | {texto_placa}'
                placas_detectadas.append({'frame': frame_count, 'texto': texto_placa, 'confianca': conf})

            # Fundo escuro para o texto
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
            cv2.rectangle(frame_anotado, (x1, y1 - th - 12), (x1 + tw + 8, y1), (0, 0, 0), -1)
            cv2.putText(frame_anotado, label, (x1 + 4, y1 - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            # ── Alerta sonoro (a cada N segundos) ──
            agora = time.time()
            if texto_placa and (agora - ultimo_alerta) > intervalo_alerta:
                print(f'🔔 Placa detectada no frame {frame_count}: {texto_placa} (conf: {conf:.0%})')
                alerta_sonoro()
                ultimo_alerta = agora

        out.write(frame_anotado)

        # Mostra frame no Colab a cada 30 frames
        if mostrar_frames and frame_count % 30 == 0:
            print(f'Frame {frame_count}/{total_frames}')
            cv2_imshow(frame_anotado)

    cap.release()
    out.release()

    print(f'\n✅ Processamento concluído!')
    print(f'📊 Total de detecções com OCR: {len(placas_detectadas)}')
    if placas_detectadas:
        print('\n📋 Placas encontradas:')
        for p in placas_detectadas:
            print(f"  Frame {p['frame']}: {p['texto']} ({p['confianca']:.0%})")

    return placas_detectadas

# ── Rodar detecção ────────────────────────────────────────────────────────────
placas = detectar_placas_video(
    video_path,
    confianca=0.5,
    mostrar_frames=True
)

## 7️⃣ Baixar o vídeo com as detecções

In [ ]:
from google.colab import files
files.download('resultado_deteccao.mp4')
print('✅ Download iniciado!')

## 8️⃣ (Opcional) Salvar modelo no Google Drive

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
shutil.copy(
    'runs/detect/placas_caminhao/weights/best.pt',
    '/content/drive/MyDrive/modelo_placas.pt'
)
print('✅ Modelo salvo no Google Drive!')